# 01 · Limpieza y calidad del catálogo
**EP5 · tratamiento explícito de valores faltantes y duplicados**

Este notebook ejecuta las reglas de `src/pipeline.py`; no duplica transformaciones manuales ni crea archivos intermedios. Todas las decisiones pueden reproducirse ejecutando los notebooks desde la raíz del proyecto.

**Política:** preservar nulos; filtrar categorías faltantes *solo en el gráfico correspondiente*. No inventar país, director, género o clasificación etaria.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'pipeline.py').exists() else Path.cwd().parent
assert (ROOT / 'src' / 'pipeline.py').exists(), 'Ejecuta Jupyter desde la raíz del proyecto o notebooks/'
sys.path.insert(0,str(ROOT))
import pandas as pd, numpy as np
from IPython.display import display
from src.pipeline import load_sources, clean, audit, kpis, looker_table, OUT


## 1. Diagnóstico antes y después

In [2]:
raw=load_sources()
df=clean(raw)
print(f"Filas originales: {len(raw):,}; contenidos únicos: {len(df):,}; consolidados: {len(raw)-len(df)}")
display(audit(raw).sort_values("porcentaje",ascending=False).head(12))
assert df.content_key.is_unique
assert len(df)==31991

Filas originales: 32,000; contenidos únicos: 31,991; consolidados: 9


,campo,faltantes,porcentaje
16,budget,16000,50.00
9,duration,16000,50.00
17,revenue,16000,50.00
3,director,11097,34.68
12,description,3338,10.43
5,country,2263,7.07
4,cast,1361,4.25
10,genres,1081,3.38
1,type,0,0.00
0,show_id,0,0.00


## 2. Cobertura real de las dimensiones analíticas

In [3]:
campos=["genero_principal","pais_principal","director_principal","nota_confiable_50","presupuesto_usd"]
coverage=df.groupby("type")[campos].apply(lambda g:(g.notna().mean()*100).round(1))
display(coverage)
print("Contenido sin género:",df.genero_principal.isna().sum())
print("Contenido sin país:",df.pais_principal.isna().sum())

,genero_principal,pais_principal,director_principal,nota_confiable_50,presupuesto_usd
type,,,,,
Movie,99.3,97.1,99.2,76.2,21.8
TV Show,93.9,88.8,31.5,20.1,0.0


Contenido sin género: 1081
Contenido sin país: 2261


## 3. Taxonomía de géneros
Conservamos el género principal como categoría **mutuamente exclusiva** para Looker. Se guarda además la lista armonizada, pero no se cuenta como una tabla expandida. 

No fusionamos ciegamente `Action` con `Action & Adventure`, ni `War` con `War & Politics`: combinarlos implicaría perder información. `Unknown` se convierte en nulo; no en un género válido.

In [4]:
display(df.genero_principal.value_counts(dropna=False).head(20))
print("Géneros principales distintos:",df.genero_principal.nunique())

genero_principal
Drama                         8667
Comedy                        4681
Animation                     3112
Acción                        1883
Horror                        1520
Documentary                   1516
Reality                       1349
Crime                         1135
<NA>                          1081
Thriller                       961
Romance                        914
Family                         779
Acción y aventura              677
Mystery                        513
Talk                           450
Ciencia ficción                416
Aventura                       381
Ciencia ficción y fantasía     373
Soap                           339
Fantasía                       247
Name: count, dtype: int64

Géneros principales distintos: 27


## 4. Votos y calidad: indicador confiable
Compararemos calificaciones principalmente con un mínimo de **50 votos**; también conservamos `nota_con_votos` para evaluar cobertura. El umbral es una decisión de análisis, no una garantía de representatividad.

In [5]:
display(df.groupby("type").agg(titulos=("content_key","size"),con_1_voto=("nota_con_votos","count"),con_50_votos=("nota_confiable_50","count"),nota_media_50=("nota_confiable_50","mean")).round(3))
print("Series con 1–9 votos:",((df.type=="TV Show") & df.vote_count.between(1,9)).sum())

,titulos,con_1_voto,con_50_votos,nota_media_50
type,,,,
Movie,16000,15106,12185,6.390
TV Show,15991,12325,3212,7.519


Series con 1–9 votos: 6177


## 5. Finanzas: definiciones inequívocas
`finanzas_validas`: solo películas con `budget >= 10.000` y `revenue >= 10.000`, según criterio histórico del proyecto (umbral de análisis, no verificación externa).

- **Múltiplo ingresos/presupuesto** = ingresos / presupuesto, expresado como ×.
- **ROI neto aproximado** = (ingresos − presupuesto) / presupuesto. No representa beneficio neto empresarial ni costos de licencias.
- **Recuperación de presupuesto** = ingresos >= presupuesto. No equivale a rentabilidad integral de la plataforma.
- Las series no se consideran en cálculos financieros.

In [6]:
print(pd.Series(kpis(df)).to_string())
f=df[df.finanzas_validas]
display(f.groupby("rango_presupuesto",observed=True).agg(n=("content_key","size"),multiplo_mediano=("multiplo_ingresos","median"),pct_recupera=("recupera_presupuesto",lambda s:100*s.mean())).round(2))

titulos_unicos                          3.199100e+04
peliculas                               1.600000e+04
series                                  1.599100e+04
n_con_50_votos                          1.539700e+04
nota_media_50_votos                     6.626000e+00
popularidad_media_total_solo_control    4.262600e+01
peliculas_financieras                   3.492000e+03
multiplo_ingresos_agregado              2.924100e+00
multiplo_ingresos_mediano               1.719600e+00
pct_recupera_presupuesto                6.386000e+01
ganancia_media_usd                      6.922299e+07


,n,multiplo_mediano,pct_recupera
rango_presupuesto,,,
< 1 M,132,2.52,72.73
1–10 M,1108,1.34,56.77
10–50 M,1511,1.44,59.10
50–100 M,367,2.21,77.66
> 100 M,374,2.71,87.43


## Hallazgos depurados (para utilizar en storytelling)
- **31.991 contenidos únicos** después de consolidar nueve pares de registros con mismo tipo/ID. En cada par se conserva el registro con mayor número de votos y, ante empate, mayor popularidad; esto es una regla reproducible, **no** prueba que dicho valor sea más reciente.
- **Ausencias reales** de país, género y especialmente director en series. En gráficos temáticos se excluyen nulos de la categoría, pero se comunica su cobertura.
- En series hay numerosos títulos con muy pocos votos. La comparación de notas con `>=50` votos no representa a todas las series.
- En películas con información financiera analizable, el indicador de recuperación de presupuesto es próximo a **64%**, no 99,88%.
- La popularidad TMDb no equivale a visualizaciones o suscriptores; se usa en análisis separados por formato o mediante percentiles dentro del formato.

**Limitación de deduplicación:** una fuente con historial temporal permitiría elegir la versión más reciente; no se dispone de ella.